# <font color="#003660">Applied Machine Learning for Text Analysis (M.184.5331)</font>

# <font color="#003660">Session 4: Prompt Engineering</font>

# <font color="#003660">Notebook 3: Prompting with the OpenAI-Compatible API</font>

<center><br><img width=256 src="https://raw.githubusercontent.com/olivermueller/aml4ta-2021/main/resources/dag.png"/><br></center>

<p>

<div>
    <font color="#085986"><b>By the end of this lesson, you ...</b><br><br>
        ... know why the OpenAI-compatible API is the base for almost everything you will build with LLMs, <br>
        ... can connect to a model on your computer (Ollama), on the university's AI-Gateway, or on any other OpenAI-compatible API, <br>
        ... can send messages, system prompts, and few-shot examples with LangChain's <code>ChatOpenAI</code>, <br>
        ... can control the generation with <code>temperature</code>, <code>seed</code>, <code>max_tokens</code>, and <code>stop</code>.
    </font>
</div>
</p>

Inspired by these excellent sources:

* [Raschka (2024): Building a Large Language Model (From Scratch)](https://www.manning.com/books/build-a-large-language-model-from-scratch)
* [HuggingFace (2024): NLP Course](https://huggingface.co/learn/nlp-course/)
* [Huggingface (2024): Open-Source AI Cookbook](https://huggingface.co/learn/cookbook/index)
* [Prompt Engineering Guide](https://www.promptingguide.ai/)

Documentation for this notebook:

* [AI-Gateway of Paderborn University](https://hilfe.uni-paderborn.de/AI-Gateway)
* [OpenAI API reference: chat completions](https://platform.openai.com/docs/api-reference/chat/create?lang=python)
* [LangChain Docs: ChatOpenAI](https://docs.langchain.com/oss/python/integrations/chat/openai)
* OpenAI-compatible endpoints of other providers: [Ollama](https://docs.ollama.com/api/openai-compatibility), [Hugging Face Inference Providers](https://huggingface.co/docs/inference-providers/index?python-clients=openai&javascript-clients=openai), [vLLM](https://docs.vllm.ai/)

## Where We Are

* **4.01 and 4.02:** you wrote prompt templates by hand and used Ollama's own API.
* **Now:** the API you will use in your projects, the **OpenAI-compatible API**, with **LangChain**, the library we use for the rest of the course.

There are no new prompting concepts here. You do the same things as before, just through a different API.

## Why This API?

**In short: learn it once, use it everywhere.**

* OpenAI designed this interface for its own models. It became a de facto standard.
* **Every provider speaks it:** OpenAI, Ollama, vLLM, Hugging Face, cloud providers, and even our university.
* **Everything that comes later builds on it:** structured output, tool calling, RAG, agents. They are all requests to `chat.completions`.
* **We use it through LangChain:** `ChatOpenAI` is LangChain's chat model for the OpenAI-compatible API. Underneath, it uses the `openai` package.
* **Switching the provider means changing three things:** the address (`base_url`), the key (`api_key`), and the model name.

Using the OpenAI API *format* does not mean using OpenAI's *models*. In this notebook, you choose the provider yourself: Ollama on your computer or on Google Colab, the course's Hugging Face endpoint, the university's **AI-Gateway**, or another API.

## Setup: Where the Model Runs

**In short: you choose where the model runs. The rest of the notebook stays the same.**

| `provider` | Where the model runs | `model_name` | Good to know |
|---|---|---|---|
| `"ollama"` | **Ollama** on your computer | `qwen3.5:9b`, `qwen3.5:4b`, `qwen3.5:2b` | free, no account; install [Ollama](https://ollama.com/download) first, then download the model once in a terminal: `ollama pull qwen3.5:9b` (6.6 GB; 4b: 3.3 GB; 2b: 2.7 GB); without a GPU, it runs on the CPU, which is slower |
| `"colab"` | **Ollama on Google Colab** | the same three | free, needs a Google account; the cell installs Ollama in the Colab runtime and downloads the model, as in 4.01; choose a GPU runtime first (Runtime > Change runtime type > T4 GPU) |
| `"huggingface"` | **Hugging Face endpoints** of the course | the same three, one endpoint each | key and addresses from the course; if an endpoint was asleep, the first request takes 1 to 2 minutes |
| `"ai-gateway"` | **AI-Gateway** of Paderborn University | `gwdg.qwen3.8-27b-no-think` (thinking off), `gwdg.qwen3.8-27b` (thinking on) | your university key (see [setup/SETUP_AI_GATEWAY.md](https://github.com/olivermueller/amlta-2026/blob/main/setup/SETUP_AI_GATEWAY.md)); only in the university network or the VPN, not on Colab; about 10 requests per minute |
| `"other"` | **another** OpenAI-compatible API | the name your provider uses | base URL and key in `.env` (`OTHER_BASE`, `OTHER_KEY`) |

Keys and addresses come from the `.env` file in the repo folder (see `.env.example`). Without a `.env` file, for example on Colab, the cell asks for them.

The providers `ollama`, `colab`, `huggingface`, and `other` need nothing else. For `ai-gateway`, you first need access to the gateway and a key: see [setup/SETUP_AI_GATEWAY.md](https://github.com/olivermueller/amlta-2026/blob/main/setup/SETUP_AI_GATEWAY.md).

### Connect from Python

We install `langchain-openai`, LangChain's package for the OpenAI-compatible API. It brings the `openai` package with it.

In [ ]:
%pip install -q langchain-openai python-dotenv

The next cell defines `setup_model()`. The cell after it calls it with your choice, and sets three values:

| Name | What it is |
|---|---|
| `BASE_URL` | the address of the provider |
| `API_KEY` | your key (Ollama needs none) |
| `MODEL_NAME` | the model |

Keys come from the `.env` file: `load_dotenv()` looks for it in the notebook's folder and then in the folders above, so it finds the file in the repo folder.

In [ ]:
# Run this cell: it only defines setup_model(). We call it before every ChatOpenAI, with the provider and the model.
import getpass
import json
import os
import shutil
import subprocess
import sys
import time
import urllib.error
import urllib.request

from dotenv import load_dotenv

load_dotenv()  # reads the .env file if there is one: this folder first, then the folders above (the repo folder)
IN_COLAB = "google.colab" in sys.modules

PROVIDERS = ["ollama", "colab", "huggingface", "ai-gateway", "other"]
QWEN = {  # model_name: (download size in Ollama, the name of its endpoint address in .env, the model on Hugging Face)
    "qwen3.5:9b": ("6.6 GB", "HF_GATEWAY_BASE_9B", "unsloth/Qwen3.5-9B-GGUF"),
    "qwen3.5:4b": ("3.3 GB", "HF_GATEWAY_BASE_4B", "unsloth/Qwen3.5-4B-GGUF"),
    "qwen3.5:2b": ("2.7 GB", "HF_GATEWAY_BASE_2B", "unsloth/Qwen3.5-2B-GGUF"),
}
GATEWAY_QWEN = {  # on the AI-Gateway, the model name decides about thinking
    "gwdg.qwen3.8-27b-no-think": "thinking off",
    "gwdg.qwen3.8-27b": "thinking on",
}


def value(env_name, question, secret=False):
    """Takes the value from .env, or asks for it (keys are typed hidden)."""
    found = os.getenv(env_name)
    if found:
        return found.strip()
    answer = getpass.getpass(question).strip() if secret else input(question).strip()
    os.environ[env_name] = answer  # asked once: the next call of setup_model() takes it from here
    return answer


def start_ollama(model, size, colab=False):
    """Makes sure that the Ollama server runs and has the model. Downloads the model only on Colab."""
    url = (os.getenv("OLLAMA_BASE") or "http://localhost:11434").rstrip("/")

    def running():
        try:
            urllib.request.urlopen(url, timeout=2)
            return True
        except OSError:
            return False

    if colab and shutil.which("ollama") is None:
        # the same steps as in 4.01: the installer needs pciutils and zstd
        print("Installing Ollama in the Colab runtime (about a minute) ...")
        subprocess.run("sudo apt-get install -y -qq pciutils zstd > /dev/null && curl -fsSL https://ollama.com/install.sh | sh > /dev/null",
                       shell=True, check=True)
    if not running():
        if shutil.which("ollama") is None:
            raise RuntimeError("Ollama is not installed. Install it from https://ollama.com/download and run this cell again.")
        # on Colab, the installer may say that the server is running, but it is not; so we start it ourselves (as in 4.01)
        print("Starting the Ollama server ...")
        ollama_log = open("ollama.log", "w")  # a pipe that nobody reads could fill up and block the server
        subprocess.Popen(["ollama", "serve"], stdout=ollama_log, stderr=ollama_log)
        for _ in range(30):
            if running():
                break
            time.sleep(1)
        else:
            raise RuntimeError("The Ollama server did not start. Look into ollama.log.")

    installed = [m["name"] for m in json.load(urllib.request.urlopen(url + "/api/tags"))["models"]]
    if model not in installed and not colab:  # on your computer, you download the model yourself, once
        raise RuntimeError(f"{model} is not downloaded yet. Run this command in a terminal (it downloads {size}), "
                           f"then run this cell again:\n\n    ollama pull {model}\n")
    if model not in installed:
        print(f"WARNING: {model} is not here yet. It is downloaded now ({size}), this can take several minutes.")
        request = urllib.request.Request(url + "/api/pull", data=json.dumps({"model": model}).encode())
        shown = -1
        for line in urllib.request.urlopen(request):
            status = json.loads(line)
            if "error" in status:
                raise RuntimeError(status["error"])
            if status.get("total"):
                percent = int(100 * status.get("completed", 0) / status["total"]) // 10 * 10
                if percent != shown:
                    print(f"  {percent} %", end="  ")
                    shown = percent
        print("\nDownload finished.")
    return url + "/v1/"


def wait_until_awake(base_url, api_key):
    """A sleeping Hugging Face endpoint answers 503 until it has started; we wait for it."""
    request = urllib.request.Request(base_url + "/models", headers={"Authorization": f"Bearer {api_key}"})
    for attempt in range(40):  # up to about 6 minutes
        try:
            urllib.request.urlopen(request, timeout=10)
            return
        except urllib.error.HTTPError as error:
            if error.code == 403:
                raise RuntimeError("The endpoint answers 403: it is paused (the course endpoints only run during the seminar), "
                                   "or your key has no access to it. Choose another provider, or check HF_GATEWAY_KEY and the address in .env.")
            if error.code != 503:
                raise RuntimeError(f"The endpoint answers {error.code}. Check the key and the address.")
        except OSError:
            pass
        if attempt == 0:
            print("The endpoint is asleep and starts now. This takes a few minutes ...")
        time.sleep(8)
    raise RuntimeError("The endpoint did not wake up. Try again in a few minutes.")


def setup_model(model):
    """Checks {"model_name": ..., "provider": ...}, sets the address and the key, and prints what to keep in mind.
    Returns base_url, api_key and the model name for ChatOpenAI."""
    name = model.get("model_name", "").strip()
    provider = model.get("provider", "").strip().lower()
    if provider not in PROVIDERS:
        raise ValueError(f"Unknown provider {provider!r}. Choose one of: {', '.join(PROVIDERS)}.")
    if provider in ("ollama", "colab", "huggingface") and name not in QWEN:
        raise ValueError(f"The provider {provider!r} has these models: {', '.join(QWEN)}. Not {name!r}.")
    notes = []

    if provider in ("ollama", "colab"):
        if provider == "ollama" and IN_COLAB:
            raise ValueError("This notebook runs on Google Colab: use the provider 'colab'.")
        if provider == "colab" and not IN_COLAB:
            raise ValueError("The provider 'colab' only works on Google Colab. On your own computer, use 'ollama'.")
        if provider == "colab" and subprocess.run("nvidia-smi", shell=True, capture_output=True).returncode != 0:
            notes.append("This Colab runtime has no GPU: answers take long. Better: Runtime > Change runtime type > T4 GPU, then run the notebook again.")
        if provider == "ollama":
            notes.append("The model runs on this computer. Without a GPU, it runs on the CPU, and answers take longer.")
        base_url, api_key = start_ollama(name, QWEN[name][0], colab=(provider == "colab")), "ollama"  # Ollama needs no key
        notes.append('Thinking: switch it with "reasoning_effort" in the extra_body of ChatOpenAI ("none" = off).')
        notes.append("Several answers per request (n > 1): Ollama returns only one.")

    elif provider == "huggingface":
        _, env_name, name = QWEN[name]  # name: the model on Hugging Face
        base_url = value(env_name, f"Paste the address of the Hugging Face endpoint for {model['model_name']}: ").rstrip("/")
        if "<" in base_url:
            raise ValueError(f"The address for {model['model_name']} is still a placeholder in .env ({env_name}). "
                             "Paste the address from the course there, or choose another model or provider.")
        if not base_url.endswith("/v1"):
            base_url += "/v1"  # the OpenAI-compatible API of the endpoint
        api_key = value("HF_GATEWAY_KEY", "Paste your Hugging Face key (for a public endpoint, just press Enter): ", secret=True)
        api_key = api_key or "none"  # a public endpoint needs no key, but the client needs some text
        wait_until_awake(base_url, api_key)
        notes.append('Thinking: switch it with "reasoning_effort" in the extra_body of ChatOpenAI ("none" = off).')
        notes.append("Several answers per request (n > 1): the endpoint refuses them (HTTP 400).")

    elif provider == "ai-gateway":
        if IN_COLAB:
            notes.append("Colab runs outside the university network and usually cannot reach the AI-Gateway.")
        base_url = os.getenv("AI_GATEWAY_BASE") or "https://ai-gateway.uni-paderborn.de/v1"
        api_key = value("AI_GATEWAY_KEY", "Paste your AI-Gateway key: ", secret=True)
        if name not in GATEWAY_QWEN:
            notes.append(f"{name!r} is not one of the Qwen models of the course ({', '.join(GATEWAY_QWEN)}). "
                         "llm.root_client.models.list() lists the models of your account.")
        notes.append("Thinking cannot be switched with reasoning_effort here (the gateway ignores it): the model name decides. "
                     "gwdg.qwen3.8-27b-no-think = off, gwdg.qwen3.8-27b = on.")
        notes.append("Only in the university network or the VPN. About 10 requests per minute: if a cell pauses, the client waits and continues.")

    else:  # another OpenAI-compatible API
        if not name:
            raise ValueError("Give the model name of your provider as model_name.")
        base_url = value("OTHER_BASE", "Base URL of the API (e.g. https://api.openai.com/v1/): ")
        api_key = value("OTHER_KEY", "Your API key: ", secret=True)
        notes.append("Check in the documentation of your provider how thinking is switched; "
                     "the extra_body of ChatOpenAI may need other fields than reasoning_effort.")

    print(f"Provider: {provider} | model: {name}")  # the key is never printed
    print("Keep in mind:")
    for note in notes:
        print("  -", note)
    return base_url, api_key, name

**Before every `ChatOpenAI`, we call `setup_model()`** with the provider and the model, for example `setup_model({"model_name": "qwen3.5:9b", "provider": "huggingface"})`. It checks whether the combination exists, sets the address and the key, and prints what to keep in mind. It returns three values, which we split directly into `BASE_URL`, `API_KEY`, and `MODEL_NAME`. To switch the model, change the dict there.

`ChatOpenAI` is LangChain's chat model for the OpenAI-compatible API. Everything about the model is set here, before the first request: the provider (`base_url`, `api_key`), the model, `temperature`, and in `extra_body` the token limit and whether the model thinks (Part 4). You can change single settings for one request (Part 3).

In [ ]:
from langchain_openai import ChatOpenAI

BASE_URL, API_KEY, MODEL_NAME = setup_model({"model_name": "qwen3.5:9b", "provider": "huggingface"})  # provider and model: see the table in the setup
llm = ChatOpenAI(
    model=MODEL_NAME,  # on the AI-Gateway, the model decides about thinking: gwdg.qwen3.8-27b-no-think (off) or gwdg.qwen3.8-27b (on)
    base_url=BASE_URL,  # your provider instead of OpenAI's servers
    api_key=API_KEY,
    temperature=0,
    extra_body={
        "max_tokens": 512,  # the token limit (here, because Ollama ignores max_completion_tokens, the name that ChatOpenAI uses)
        "reasoning_effort": "none",  # thinking off; "low", "medium", or "high" switches it on (Ollama, Hugging Face endpoint)
    },
)

**Never write the key into the notebook itself.** Notebooks end up on GitHub; the `.env` file does not, because it is listed in `.gitignore`.

### Check the connection

The provider lists the models you can use. Underneath, `ChatOpenAI` uses the `openai` package: `llm.root_client` is its client, and it can list the models. An error here? With the gateway, check the VPN and the key. With Ollama, check that it is running.

In [ ]:
for model in llm.root_client.models.list():
    print(model.id)

`setup_model()` set the model, in `MODEL_NAME`:

* On the **gateway**, it is **Qwen3.8** with 27 billion parameters. The `-no-think` at the end of the name means that thinking (4.02, Part 3) is switched off: the model answers right away.
* With **Ollama** and the **Hugging Face endpoint**, it is **Qwen3.5**, and `"reasoning_effort": "none"` in `extra_body` switches the thinking off.

You can try other models from the list later: create a new `ChatOpenAI` with another id as `model`.

### One small helper

`score()` works like in 4.01 and 4.02, but with messages instead of a raw template:

* It fills the placeholder `{text}` in every message with each example.
* It sends the messages with `llm.invoke()`.

`limit(max_tokens)` sets the token limit of one request. Why it goes into `extra_body` is explained in Part 4.
* It counts the correct answers.

**Good to know:** `score()` sends one request per example. Some models on the gateway have a limit of requests per minute. If a cell pauses, the client is waiting and then continues by itself.

In [ ]:
import re

GREEN, ORANGE, RESET = "\033[92m", "\033[33m", "\033[0m"


def check(answer, expected):
    """Looks for the expected answer behind "The answer is" or, if the model did not write that, in the first line."""
    answer = answer.split("</think>")[-1]  # ignore the reasoning of reasoning models
    position = answer.lower().find("answer is")
    if position >= 0:
        answer = answer[position + len("answer is"):]
    first_line = next((line.strip() for line in answer.splitlines() if line.strip()), "")
    correct = re.search(rf"\b{re.escape(expected)}\b", first_line, re.IGNORECASE) is not None
    return correct, first_line


def fill(messages, text):
    """Replaces the placeholder {text} in every message."""
    return [{**message, "content": message["content"].replace("{text}", text)} for message in messages]


def limit(max_tokens):
    """The extra_body of llm with another token limit, for one request (why the limit goes into extra_body: 4.03, Part 4)."""
    return {**llm.extra_body, "max_tokens": max_tokens}


def score(messages, examples, temperature=0, max_tokens=50, **params):
    """Sends the messages for every example and counts the correct answers."""
    correct, prompt_tokens, tokens = 0, 0, 0
    for text, expected in examples:
        response = llm.invoke(fill(messages, text), temperature=temperature, extra_body=limit(max_tokens), **params)
        ok, answer = check(response.content or "", expected)
        correct += ok
        prompt_tokens += response.usage_metadata["input_tokens"]
        tokens += response.usage_metadata["output_tokens"]
        mark = f"{GREEN}✓{RESET}" if ok else f"{ORANGE}✗{RESET}"
        print(f"{mark} expected: {expected:<10} model: {answer[:70]}")
    print(f"\n{correct}/{len(examples)} correct, {prompt_tokens} prompt tokens, {tokens} tokens generated")

## Part 1: One Request

You know the messages format from 4.02. `llm.invoke()` takes exactly the same list:

In [ ]:
messages = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "What is the capital of France?"},
]

response = llm.invoke(messages)
print(response.content)

The API answers with a JSON object, and LangChain turns it into an `AIMessage`. Besides the answer in `content`, look for two things:

* `finish_reason` in `response_metadata`: why the generation stopped
* `usage_metadata`: how many tokens were used

In [ ]:
print("finish_reason:", response.response_metadata["finish_reason"])
print("usage:", response.usage_metadata)
print()
print(response.response_metadata)  # everything else the API sent back

Behind the scenes, the server does what you did by hand in 4.02: it puts your messages into the model's **chat template**. The API is just a more convenient way to get to the same template.

## Part 2: System Prompts and Few-Shot Examples as Messages

**In short: system prompts and few-shot examples are just messages.**

* **System prompt:** the first message in the list.
* **Few-shot examples:** earlier `user` and `assistant` messages that you write yourself. The model sees them as earlier turns of the conversation.

Let's do the sentiment task from 4.01:

In [ ]:
# ten short movie reviews with their sentiment
REVIEWS = [
    ("A wonderful film with brilliant actors.", "positive"),
    ("The plot was boring and far too long.", "negative"),
    ("I laughed from start to finish.", "positive"),
    ("A complete waste of two hours.", "negative"),
    ("Not bad at all, I would watch it again.", "positive"),
    ("The ending made no sense, and the dialogue was awful.", "negative"),
    ("Beautiful music and a touching story.", "positive"),
    ("I fell asleep halfway through.", "negative"),
    ("It was not as good as everyone said.", "negative"),
    ("My kids loved it, and so did I.", "positive"),
]

In [ ]:
zero_shot = [
    {"role": "system", "content": "Classify the sentiment of the movie review. Answer with positive or negative only."},
    {"role": "user", "content": "{text}"},
]
score(zero_shot, REVIEWS)

### Your turn

1. Add one or more examples as earlier `user` and `assistant` messages. Use your own reviews, not reviews from the list.
2. Compare the score with your best base-model template from 4.01. Does this model need the examples?

In [ ]:
# TODO: add your examples as earlier user and assistant messages
few_shot = [
    {"role": "system", "content": "Classify the sentiment of the movie review. Answer with positive or negative only."},
    {"role": "user", "content": "{text}"},
]
score(few_shot, REVIEWS)

## Part 3: Controlling the Generation

**In short: the decoding parameters from Session 3 are part of every request.**

| Parameter | What it does |
|---|---|
| `temperature` | flattens (> 1) or sharpens (< 1) the next-token distribution; `0` means greedy decoding |
| `top_p` | samples only from the most likely tokens that together reach this probability |
| `seed` | fixes the random numbers, so that sampling gives the same result again |
| `max_tokens` | the maximum number of generated tokens |
| `stop` | a list of stop sequences, as in 4.01 |

In LangChain, you pass them to `invoke()` for one request, for example `llm.invoke(messages, temperature=1.0, seed=1)`. Only the token limit goes through `limit()`: `extra_body=limit(60)`.

`finish_reason` (in `response.response_metadata`) tells you why the generation stopped: `length` (maximum number of tokens) or `stop` (end-of-turn token or stop sequence).

In [ ]:
messages = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "Write a short poem about the sea."},
]

for seed in [1, 2, 3]:
    response = llm.invoke(messages, temperature=1.0, seed=seed, extra_body=limit(60))
    print(f"seed {seed}, finish_reason: {response.response_metadata["finish_reason"]}")
    print(response.content, end="\n\n")

### Your turn

**Predict** before you run it:

* What changes with `temperature=0`?
* What is the `finish_reason` with `max_tokens=200`?
* And with `stop=["\n"]`?

In [ ]:
# TODO: change temperature, max_tokens, and stop, and compare the outputs and the finish_reason


Not every provider supports every parameter, for example `seed`. If a parameter is ignored or rejected, check the provider's documentation.

## Part 4: Switching the Provider

**In short: change `base_url`, `api_key`, and the model name. The rest of your code stays.**

Some examples (not to run now):

```python
import os

# Ollama on your own computer, as in 4.01 and 4.02 (Ollama does not check the key, but the package needs one)
llm = ChatOpenAI(model="qwen3.5:2b", base_url="http://localhost:11434/v1", api_key="ollama")

# a cloud provider, for example Hugging Face Inference Providers (needs a Hugging Face token)
llm = ChatOpenAI(model="<model id>", base_url="https://router.huggingface.co/v1", api_key=os.environ["HF_TOKEN"])

# a vLLM server that you started yourself
llm = ChatOpenAI(model="<model id>", base_url="http://localhost:8000/v1", api_key="EMPTY")

# OpenAI itself: the key is read from the environment variable OPENAI_API_KEY
llm = ChatOpenAI(model="<model id>")
```

Three things still differ between providers:

* **The template.** Every model family has its own, and the provider applies it for you.
* **The parameters.** Not every provider supports every parameter, or every name of it. `ChatOpenAI` sends `max_tokens` under its new name `max_completion_tokens`, and Ollama ignores that name. So `limit()` sends the old name `max_tokens` in `extra_body`, which every provider understands. The Hugging Face endpoint refuses `n` (several answers in one request, 4.04), and not every provider supports `seed`.
* **Thinking.** Ollama and our Hugging Face endpoint (llama.cpp) switch it with `reasoning_effort` in `extra_body`: `"none"` is off, `"low"`, `"medium"`, or `"high"` is on. Our gateway uses the model name instead: `gwdg.qwen3.8-27b` thinks, `gwdg.qwen3.8-27b-no-think` does not. `extra_body=` adds fields to the request that the `openai` package does not know.

**Never write an API key into a notebook.** Read it from the `.env` file, as in the setup above, or from an environment variable.

## Questions?

Anything unclear about the OpenAI-compatible API, `ChatOpenAI`, or the parameters? Ask now.

## What You Learned

* The **OpenAI-compatible API** is a de facto standard. Structured output, tool calling, RAG, and agents all build on it.
* The same code works with the university's **AI-Gateway**, Ollama, vLLM, Hugging Face, OpenAI, and many others. You switch with `base_url`, `api_key`, and the model name.
* LangChain's `ChatOpenAI` sends a **list of messages** to the API and returns an `AIMessage`. The provider turns the messages into the model's **chat template**.
* System prompts and few-shot examples are just messages: the system message comes first, and examples are earlier `user` and `assistant` turns.
* `temperature`, `top_p`, `seed`, `max_tokens`, and `stop` control the generation, and `finish_reason` tells you why it stopped.

Next: **4.04 - Prompt Engineering Best Practices**. A few habits for writing prompts and checking them.